In [ ]:
import gurobipy as gp
from gurobipy import GRB
import math

VENUES = {
    "Toronto": (43.6333, -79.4186),
    "Vancouver": (49.282729, -123.120738),
    "Guadalajara": (20.659698, -103.349609),
    "Mexico City": (19.432608, -99.133209),
    "Monterrey": (25.686613, -100.316116),
    "Atlanta": (33.755000, -84.401000),
    "Boston": (42.0911, -71.2644),
    "Dallas": (32.747841, -97.093628),
    "Houston": (29.684700, -95.410700),
    "Kansas City": (39.04889, 94.48389),
    "Los Angeles": (33.953300, -118.338700),
    "Miami": (25.958000, -80.238900),
    "New York/New Jersey": (40.7128, -74.0060),
    "Philadelphia": (39.900800, -75.167500),
    "Seattle": (47.595152, -122.331639),
    "San Francisco Bay Area": (37.403263, -121.969688)
}

TIMEZONES = {
    "Toronto": -4,
    "Vancouver": -7,
    "Guadalajara": -6,
    "Mexico City": -6,
    "Monterrey": -6,
    "Atlanta": -4,
    "Boston": -4,
    "Dallas": -5,
    "Houston": -5,
    "Kansas City": -5,
    "Los Angeles": -7,
    "Miami": -4,
    "New York/New Jersey": -4,
    "Philadelphia": -4,
    "Seattle": -7,
    "San Francisco Bay Area": -7
}

# should replace with the optimized groups!
GROUPS = {
    "A": [
        "Mexico",
        "South Africa",
        "Korea Republic",
        "Czechia"
    ],
    "B": [
        "Canada",
        "Bosnia & Herzegovina",
        "Qatar",
        "Switzerland"
    ],
    "C": [
        "Brazil",
        "Morocco",
        "Haiti",
        "Scotland"
    ],
    "D": [
        "USA",
        "Paraguay",
        "Australia",
        "Türkiye"
    ],
    "E": [
        "Germany",
        "Curaçao",
        "Côte d'Ivoire",
        "Ecuador"
    ],
    "F": [
        "Netherlands",
        "Japan",
        "Sweden",
        "Tunisia"
    ],
    "G": [
        "Belgium",
        "Egypt",
        "IR Iran",
        "New Zealand"
    ],
    "H": [
        "Spain",
        "Cabo Verde",
        "Saudi Arabia",
        "Uruguay"
    ],
    "I": [
        "France",
        "Senegal",
        "Iraq",
        "Norway"
    ],
    "J": [
        "Argentina",
        "Algeria",
        "Austria",
        "Jordan"
    ],
    "K": [
        "Portugal",
        "Congo DR",
        "Uzbekistan",
        "Colombia"
    ],
    "L": [
        "England",
        "Croatia",
        "Ghana",
        "Panama"
    ]
}

ROUNDS = [1, 2, 3]

# kind of arbitrarily picked 5 and 3 but host countries have the highest weights and countries with large fan bases have higher weights
FAN_WEIGHTS = {
    "Mexico": 5, "Canada": 5, "USA": 5,
    "Germany": 3, "England": 3, "Brazil": 3, "Spain": 3, "Portugal": 3, "Argentina": 3, "Colombia": 3,
    "South Africa": 1, "Korea Republic": 1, "Czechia": 1,
    "Bosnia & Herzegovina": 1, "Qatar": 1, "Switzerland": 1,
    "Morocco": 1, "Haiti": 1, "Scotland": 1,
    "Paraguay": 1, "Australia": 1, "Türkiye": 1,
    "Curaçao": 1, "Côte d'Ivoire": 1, "Ecuador": 1,
    "Netherlands": 1, "Japan": 1, "Sweden": 1, "Tunisia": 1,
    "Belgium": 1, "Egypt": 1, "IR Iran": 1, "New Zealand": 1,
    "Cabo Verde": 1, "Saudi Arabia": 1, "Uruguay": 1,
    "France": 1, "Senegal": 1, "Iraq": 1, "Norway": 1,
    "Algeria": 1, "Austria": 1, "Jordan": 1,
    "Congo DR": 1, "Uzbekistan": 1,
    "Croatia": 1, "Ghana": 1, "Panama": 1
}

HOST_TEAMS = {
    "Canada": ["Toronto", "Vancouver"],
    "Mexico": ["Guadalajara", "Mexico City", "Monterrey"],
    "USA": ["Atlanta", "Boston", "Dallas", "Houston", "Kansas City", "Los Angeles", "Miami", 
            "New York/New Jersey", "Philadelphia", "San Francisco Bay Area", "Seattle"]
}

def haversine(coord1, coord2):
    lat1, lon1 = coord1
    lat2, lon2 = coord2
    R = 6371 # earth's radius in km
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    delta_phi = math.radians(lat2 - lat1)
    delta_lambda = math.radians(lon2 - lon1)
    a = math.sin(delta_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(delta_lambda / 2) ** 2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c

def generate_matches(groups):
    matches = []
    for group, teams in groups.items():
        a, b, c, d = teams
        matches.extend([(group, a, b, 1), (group, c, d, 1),
                        (group, a, c, 2), (group, b, d, 2),
                        (group, a, d, 3), (group, b, c, 3)])
    return matches

def solve_travel_times(VENUES, TIMEZONES, GROUPS, ROUNDS, HOST_TEAMS, TIMEZONE_WEIGHT=500, MAX_TRAVEL=4500, FAN_WEIGHT=0.1):
    model = gp.Model("optimize_travel_times")

    # need this to prevent the model from running for too long
    model.Params.TimeLimit = 1800
    model.Params.MIPGap = 0.01
    model.Params.MIPFocus = 1

    # create lists for venues, matches, match IDs, and teams
    venues = list(VENUES)
    matches = generate_matches(GROUPS)
    match_ids = range(len(matches))
    teams = [team for group in GROUPS.values() for team in group]
    venue_min = {v: 3 for v in venues}

    # map each team to the matches they play
    team_matches = {team: [] for team in teams}
    for m, (_, team1, team2, r) in enumerate(matches):
        team_matches[team1].append(m)
        team_matches[team2].append(m)

    # dictionaries to store distances and timezone differences between venues
    distances = {(v1, v2): haversine(VENUES[v1], VENUES[v2]) for v1 in venues for v2 in venues}
    timezone_diffs = {(v1, v2): abs(TIMEZONES[v1] - TIMEZONES[v2]) for v1 in venues for v2 in venues}

    # decision variables
    x = model.addVars(match_ids, venues, vtype=GRB.BINARY, name="x")
    z = model.addVars(teams, [1, 2], venues, venues, vtype=GRB.BINARY, name="z")

    # constraints

    # each host team muust play their first match in one of their host venues
    for team, host_venues in HOST_TEAMS.items():
        m = team_matches[team][0]
        model.addConstr(gp.quicksum(x[m, v] for v in host_venues) == 1)

    # each match must be assigned to exactly one venue
    for m in match_ids:
        model.addConstr(gp.quicksum(x[m, v] for v in venues) == 1)

    # each team must play exactly one match each round
    for team in teams:
        for r in ROUNDS:
            model.addConstr(gp.quicksum(x[m, v] for m in team_matches[team] if matches[m][3] == r for v in venues) == 1)

    # each venue has to host at least 3 matches (variable to change though!)
    for v in venues:
        model.addConstr(gp.quicksum(x[m, v] for m in match_ids) >= venue_min[v])

    # link z to consecutive match venues and prohibit trips exceeding MAX_TRAVEL
    for team in teams:
        for r in [1, 2]:
            m1 = next(m for m in team_matches[team] if matches[m][3] == r)
            m2 = next(m for m in team_matches[team] if matches[m][3] == r + 1)
            for v1 in venues:
                for v2 in venues:
                    model.addConstr(z[team, r, v1, v2] <= x[m1, v1])
                    model.addConstr(z[team, r, v1, v2] <= x[m2, v2])
                    model.addConstr(z[team, r, v1, v2] >= x[m1, v1] + x[m2, v2] - 1)
                    if distances[v1, v2] > MAX_TRAVEL:
                        model.addConstr(z[team, r, v1, v2] == 0)

    # objective function minimizes team travel, timezone changes, and weighted fan travel
    travel_distance = gp.quicksum(distances[v1, v2] * z[team, r, v1, v2] for team in teams for r in [1, 2] for v1 in venues for v2 in venues)
    timezone_penalty = gp.quicksum(timezone_diffs[v1, v2] * z[team, r, v1, v2] for team in teams for r in [1, 2] for v1 in venues for v2 in venues)
    fan_travel = gp.quicksum(FAN_WEIGHTS[team] * distances[v1, v2] * z[team, r, v1, v2] for team in teams for r in [1, 2] for v1 in venues for v2 in venues)
    model.setObjective(travel_distance + FAN_WEIGHT * fan_travel +TIMEZONE_WEIGHT * timezone_penalty, GRB.MINIMIZE)
    model.optimize()

    if model.SolCount == 0:
        print("No feasible solution found.")
        return None

    schedule = {r: [] for r in ROUNDS}
    team_venues = {team: [None] * len(ROUNDS) for team in teams}

    # Track which match number each group is on
    group_match_count = {group: 0 for group in GROUPS}

    print("\n--- Optimized Match Schedule ---")
    print(f"{'Group':<8} {'Match ID':<10} {'Team 1':<22} {'Team 2':<22} {'City'}")

    for m in match_ids:
        group, team1, team2, r = matches[m]
        venue = next(v for v in venues if x[m, v].X > 0.5)
        group_match_count[group] += 1
        match_id = f"{group}_{group_match_count[group]}"
        schedule[r].append((group, team1, team2, venue))
        team_venues[team1][r - 1] = venue
        team_venues[team2][r - 1] = venue
        print(f"{group:<8} {match_id:<10} {team1:<22} {team2:<22} {venue}")

    print("\n--- Optimized Team Venues ---")
    for team, venues_used in team_venues.items():
        print(f"{team}: {venues_used}")

    print(f"\nTotal Travel Distance: {travel_distance.getValue():,.0f} km")
    print(f"Total Timezone Changes: {timezone_penalty.getValue():.0f}")
    print(f"Objective Value: {model.ObjVal:,.0f}")

    return schedule, team_venues

solve_travel_times(VENUES, TIMEZONES, GROUPS, ROUNDS, HOST_TEAMS, TIMEZONE_WEIGHT=500, MAX_TRAVEL=4500, FAN_WEIGHT=0.1)